# 03 — ML Win Predictor

**Goal:** Build a machine learning model that predicts the winning team before a match starts, using pre-match features.

**Model:** Random Forest Classifier

**Features used:** Team 1, Team 2, Toss Winner, Toss Decision, Venue, Season

**Target:** Winner (team name)

In [13]:
df = matches[
    matches['result'].isin(['runs', 'wickets'])
].copy()

df = df[
    [
        'season',
        'team1',
        'team2',
        'toss_winner',
        'toss_decision',
        'venue',
        'winner'
    ]
].dropna()

print("ML dataset shape:", df.shape)
print(df.head())

ML dataset shape: (1076, 7)
    season                        team1                        team2  \
0  2007/08  Royal Challengers Bangalore        Kolkata Knight Riders   
1  2007/08              Kings XI Punjab          Chennai Super Kings   
2  2007/08               Delhi Capitals             Rajasthan Royals   
3  2007/08               Mumbai Indians  Royal Challengers Bangalore   
4  2007/08        Kolkata Knight Riders          Sunrisers Hyderabad   

                   toss_winner toss_decision  \
0  Royal Challengers Bangalore         field   
1          Chennai Super Kings           bat   
2             Rajasthan Royals           bat   
3               Mumbai Indians           bat   
4              Deccan Chargers           bat   

                                        venue                       winner  
0                       M Chinnaswamy Stadium        Kolkata Knight Riders  
1  Punjab Cricket Association Stadium, Mohali          Chennai Super Kings  
2                  

## Step 1 — Prepare Dataset

We filter matches with valid results (won by runs or wickets) and select only the features needed for prediction. Ties and no-results are excluded.

In [14]:
print("Number of teams:", df['winner'].nunique())
print("\nTeams:")
print(sorted(df['winner'].unique()))

Number of teams: 16

Teams:
['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


## Step 2 — Explore Target Distribution

We check how many unique teams appear as winners. Class imbalance (some teams winning much more than others) can affect model performance.

In [15]:
X = df.drop('winner', axis=1)
y = df['winner']

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features:
['season', 'team1', 'team2', 'toss_winner', 'toss_decision', 'venue']

Target:
winner


## Step 3 — Define Features & Target

X = all pre-match features | y = winner (target label)

In [16]:
categorical_features = [
    'team1',
    'team2',
    'toss_winner',
    'toss_decision',
    'venue'
]

numeric_features = [
    'season'
]

## Step 4 — Define Categorical & Numeric Features

All team/venue columns are categorical and need OneHotEncoding. Season is treated as a numeric feature.

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),
        (
            'numeric',
            'passthrough',
            numeric_features
        )
    ]
)

## Step 5 — Build Preprocessing Pipeline

We use `ColumnTransformer` to apply `OneHotEncoder` to categorical columns. This pipeline ensures no data leakage between train and test sets.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 860
Testing samples: 216


## Step 6 — Train/Test Split

We use 80% data for training and 20% for testing. `stratify=y` ensures each team is proportionally represented in both splits.

In [34]:
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

                             precision    recall  f1-score   support

        Chennai Super Kings       0.59      0.79      0.68        28
             Delhi Capitals       0.50      0.41      0.45        22
              Gujarat Lions       1.00      0.33      0.50         3
             Gujarat Titans       0.75      0.50      0.60         6
       Kochi Tuskers Kerala       0.00      0.00      0.00         1
      Kolkata Knight Riders       0.53      0.77      0.62        26
       Lucknow Super Giants       0.00      0.00      0.00         5
             Mumbai Indians       0.46      0.66      0.54        29
              Pune Warriors       0.00      0.00      0.00         2
               Punjab Kings       0.38      0.14      0.20        22
           Rajasthan Royals       0.55      0.55      0.55        22
     Rising Pune Supergiant       0.00      0.00      0.00         3
Royal Challengers Bengaluru       0.48      0.50      0.49        24
        Sunrisers Hyderabad      

## Step 7 — Model Evaluation

Classification report shows per-team precision, recall, and F1 score. Some teams have lower recall due to fewer training examples.

In [ ]:
feature_names = rf_pipeline.named_steps[
    'preprocessor'
].get_feature_names_out()

importances = rf_pipeline.named_steps[
    'model'
].feature_importances_

feature_importance = (
    pd.Series(importances, index=feature_names)
    .sort_values(ascending=False)
    .head(15)
)

print(feature_importance)